# Phase 2 — Shopee Google Play Multi-Route Crawling + Cleaning + Translation

**Update in this version:** the crawler now tries multiple sort/rating routes to keep collecting after the first route returns only duplicates, with a default goal of adding 90,000 more unique reviews.


## 1. Check Python and GPU

In [1]:

import sys
import torch

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {total_gb:.2f} GB")
else:
    print("No CUDA GPU detected. Crawling will still work, but translation will be much slower on CPU.")

Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Executable: c:\Users\Lenovo\Documents\Skripsii\Skripsi\.venv\Scripts\python.exe
Torch: 2.7.1+cu118
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.00 GB


## 2. Imports, local output folder, and runtime settings

In [2]:

import time
import json
import pickle
import random
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from tqdm.auto import tqdm
from google_play_scraper import reviews, Sort
from ftfy import fix_text as ftfy_fix_text
from lingua import Language, LanguageDetectorBuilder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

pd.set_option("display.max_colwidth", 160)
tqdm.pandas()

# =========================
# Main local output path
# =========================
OUTPUT_DIR = Path(r"C:\Users\Lenovo\Documents\Skripsii\Skripsi\dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Run switches
# =========================
# Set RUN_CRAWLING = False if RAW_CSV_PATH already exists and you only want to clean/translate.
RUN_CRAWLING = True
RUN_CLEAN_TRANSLATE = True

# Optional safety test for cleaning/translation:
# - Set SMOKE_TEST_ROWS = 50 to test quickly.
# - Set SMOKE_TEST_ROWS = None for the full dataset.
SMOKE_TEST_ROWS = None

# =========================
# Output files
# =========================
RAW_CSV_PATH = OUTPUT_DIR / "shopee_google_play_reviews_raw.csv"
CLEAN_TRANSLATED_PATH = OUTPUT_DIR / "shopee_google_play_reviews_clean_translated_local_gpu.csv"
TRANSLATION_MAP_PATH = OUTPUT_DIR / "shopee_google_play_reviews_translation_map_local_gpu.csv"
MINIMAL_TRANSLATED_REVIEW_CSV_PATH = OUTPUT_DIR / "shopee_reviews_id_translated_review.csv"
UNIQUE_TRANSLATED_REVIEW_CSV_PATH = OUTPUT_DIR / "shopee_reviews_id_translated_review_unique.csv"
UNIQUE_TRANSLATED_REVIEW_XLSX_PATH = OUTPUT_DIR / "shopee_reviews_id_translated_review_unique.xlsx"
DUPLICATE_TRANSLATED_REVIEW_REPORT_PATH = OUTPUT_DIR / "shopee_reviews_duplicate_translated_review_report.csv"
TOKEN_PATH = OUTPUT_DIR / "continuation_token.pkl"
METADATA_PATH = OUTPUT_DIR / "crawl_metadata.json"
RUN_SUMMARY_PATH = OUTPUT_DIR / "crawling_clean_translation_run_summary.json"
BACKUP_ZIP_PATH = OUTPUT_DIR / "shopee_crawling_clean_translation_outputs.zip"

print("Output folder:", OUTPUT_DIR)
print("Raw CSV path :", RAW_CSV_PATH)
print("Raw exists   :", RAW_CSV_PATH.exists())

Output folder: C:\Users\Lenovo\Documents\Skripsii\Skripsi\dataset
Raw CSV path : C:\Users\Lenovo\Documents\Skripsii\Skripsi\dataset\shopee_google_play_reviews_raw.csv
Raw exists   : True


## 3. Multi-route crawling configuration

Continue **after the normal `Sort.NEWEST + all ratings` route starts returning only duplicates**.

Instead of looping the same exhausted route forever, it tries multiple safe research slices:

- `Sort.NEWEST`, `Sort.MOST_RELEVANT`, and `Sort.RATING`
- Star filters: `1`, `2`, `3`, `4`, `5`, plus `None`
- Optional language/country request pairs, while still storing the source metadata

The crawler keeps your existing raw CSV, deduplicates by `review_id_hash`, and aims to add **90,000 more unique reviews** on top of what you already collected.


In [3]:

# =========================
# Main crawling configuration
# =========================
APP_ID = "com.shopee.id"        # Shopee Indonesia on Google Play

# Existing + this many new unique reviews.
# If you already have around 90,727 reviews, this targets around 180,727 total.
TARGET_ADDITIONAL_REVIEWS = 90_000

# Minimum total target. The actual target will be:
# max(existing_unique_reviews + TARGET_ADDITIONAL_REVIEWS, MIN_TOTAL_TARGET_REVIEWS)
MIN_TOTAL_TARGET_REVIEWS = 180_000

# Google Play scraper usually works well with batches around 100-200.
BATCH_SIZE = 200

# Keep Indonesia first. "en/id" can sometimes expose a different review slice while still targeting Indonesia.
# If your lecturer wants strictly Indonesian-language request metadata, remove ("en", "id").
LANG_COUNTRY_CONFIGS = [
    ("id", "id"),
    ("en", "id"),
]

# Try star-specific routes before all-rating routes.
# Star-specific routes often expose reviews that the general route does not continue returning.
SORT_MODES_TO_TRY = [
    Sort.NEWEST,
    Sort.MOST_RELEVANT,
    Sort.RATING,
]

FILTER_SCORES_TO_TRY = [
    1, 2, 3, 4, 5, None
]

# Duplicate limits:
# - all-rating routes get skipped quickly if they are exhausted
# - star-specific routes are allowed to scan through many duplicate pages first,
#   because they may need to pass recent duplicates before reaching older unseen reviews.
MAX_DUPLICATE_BATCHES_ALL_RATINGS = 10
MAX_DUPLICATE_BATCHES_SCORE_FILTER = 180   # 180 * 200 = up to 36k duplicate reviews skipped per score route
MAX_EMPTY_BATCHES_PER_CONFIG = 3

# Polite delay between successful requests.
# This is not bot evasion; it only reduces unnecessary request pressure.
MIN_DELAY_SECONDS = 2.0
MAX_DELAY_SECONDS = 5.0

# Retry/backoff when request fails temporarily
MAX_RETRIES_PER_BATCH = 5
BASE_BACKOFF_SECONDS = 10
MAX_BACKOFF_SECONDS = 120

# Resume from previous raw CSV and per-config continuation tokens if files exist.
RESUME = True

# New multi-route token folder. This prevents the old single-route token from trapping the crawler
# at an already exhausted continuation point.
TOKEN_DIR = OUTPUT_DIR / "crawler_tokens_multi_route"
TOKEN_DIR.mkdir(parents=True, exist_ok=True)

# How often to write the full raw CSV checkpoint after adding new reviews.
SAVE_EVERY_NEW_REVIEWS = 200

print("Target additional reviews:", f"{TARGET_ADDITIONAL_REVIEWS:,}")
print("Minimum total target     :", f"{MIN_TOTAL_TARGET_REVIEWS:,}")
print("Batch size               :", BATCH_SIZE)
print("Lang/country configs     :", LANG_COUNTRY_CONFIGS)
print("Sort modes               :", [str(s) for s in SORT_MODES_TO_TRY])
print("Score filters            :", FILTER_SCORES_TO_TRY)
print("Token folder             :", TOKEN_DIR)
print("Resume                   :", RESUME)


Target additional reviews: 90,000
Minimum total target     : 180,000
Batch size               : 200
Lang/country configs     : [('id', 'id'), ('en', 'id')]
Sort modes               : ['Sort.NEWEST', 'Sort.MOST_RELEVANT', 'Sort.RATING']
Score filters            : [1, 2, 3, 4, 5, None]
Token folder             : C:\Users\Lenovo\Documents\Skripsii\Skripsi\dataset\crawler_tokens_multi_route
Resume                   : True


## 4. Crawling helper functions

The saved raw dataset excludes direct user identity columns. The original Google Play review ID is hashed into `review_id_hash` for deduplication.


In [4]:

def sha256_short(value, length=16):
    """Create a short stable hash for deduplication without storing raw identifiers."""
    value = "" if value is None else str(value)
    return hashlib.sha256(value.encode("utf-8", errors="ignore")).hexdigest()[:length]


def make_fallback_review_key(record):
    """Fallback key if reviewId is missing."""
    parts = [
        str(record.get("content", "")),
        str(record.get("score", "")),
        str(record.get("at", "")),
        str(record.get("reviewCreatedVersion", "")),
    ]
    return "|".join(parts)


def sort_name(sort_mode):
    """Make a stable readable sort name from google_play_scraper.Sort."""
    text = str(sort_mode)
    return text.split(".")[-1].lower().replace("sort.", "")


def config_name(lang, country, sort_mode, filter_score_with):
    score_part = "all" if filter_score_with is None else f"score_{filter_score_with}"
    return f"{APP_ID}_{lang}_{country}_{sort_name(sort_mode)}_{score_part}".replace(".", "_")


def token_path_for_config(lang, country, sort_mode, filter_score_with):
    return TOKEN_DIR / f"{config_name(lang, country, sort_mode, filter_score_with)}.pkl"


def normalize_review(
    record,
    app_id=APP_ID,
    lang="id",
    country="id",
    sort_mode=None,
    filter_score_with=None,
):
    """Normalize one google-play-scraper review record into safe research columns."""
    raw_review_id = record.get("reviewId") or make_fallback_review_key(record)
    at_value = record.get("at")
    replied_at_value = record.get("repliedAt")

    return {
        "review_id_hash": sha256_short(raw_review_id),
        "content": record.get("content"),
        "score": record.get("score"),
        "thumbs_up_count": record.get("thumbsUpCount"),
        "review_created_version": record.get("reviewCreatedVersion"),
        "review_at": at_value.isoformat() if hasattr(at_value, "isoformat") else at_value,
        "app_version": record.get("appVersion"),
        "reply_content": record.get("replyContent"),
        "replied_at": replied_at_value.isoformat() if hasattr(replied_at_value, "isoformat") else replied_at_value,
        "source_app_id": app_id,
        "source_lang": lang,
        "source_country": country,
        "source_sort_mode": sort_name(sort_mode) if sort_mode is not None else None,
        "source_filter_score_with": filter_score_with,
        "crawl_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }


def ensure_review_hash_column(df):
    """Make old/raw files compatible even if review_id_hash is missing."""
    if df.empty:
        return df
    if "review_id_hash" not in df.columns:
        print("review_id_hash column not found. Creating fallback hashes from content/score/date.")
        df = df.copy()
        df["review_id_hash"] = df.apply(
            lambda row: sha256_short(
                "|".join([
                    str(row.get("content", "")),
                    str(row.get("score", "")),
                    str(row.get("review_at", "")),
                    str(row.get("review_created_version", "")),
                ])
            ),
            axis=1,
        )
    return df


def load_existing_dataset(path):
    path = Path(path)
    if RESUME and path.exists():
        df = pd.read_csv(path)
        df = ensure_review_hash_column(df)
        if "review_id_hash" in df.columns:
            before = len(df)
            df = df.drop_duplicates(subset=["review_id_hash"]).reset_index(drop=True)
            if len(df) != before:
                print(f"Dropped {before - len(df):,} duplicate rows from existing dataset.")
        print(f"Loaded existing checkpoint: {len(df):,} unique rows from {path}")
        return df
    return pd.DataFrame()


def load_continuation_token(path):
    path = Path(path)
    if RESUME and path.exists():
        try:
            with open(path, "rb") as f:
                token = pickle.load(f)
            print(f"Loaded token: {path.name}")
            return token
        except Exception as e:
            print(f"Could not load token {path.name}. Starting this config without token.")
            print("Reason:", repr(e))
    return None


def save_checkpoint(df, token=None, raw_csv_path=RAW_CSV_PATH, token_path=None):
    raw_csv_path = Path(raw_csv_path)
    df.to_csv(raw_csv_path, index=False, encoding="utf-8-sig")
    if token_path is not None:
        token_path = Path(token_path)
        try:
            with open(token_path, "wb") as f:
                pickle.dump(token, f)
        except Exception as e:
            print("Warning: could not save continuation token:", repr(e))


def weak_sentiment_from_star(score):
    """Only a weak label from rating, not the final ABSA sentiment label."""
    try:
        score = int(score)
    except Exception:
        return "unknown"
    if score <= 2:
        return "negative"
    if score == 3:
        return "neutral"
    return "positive"


def build_crawl_configs():
    """Build the ordered list of crawl routes."""
    configs = []

    # First: star-specific routes, because they are the best chance to bypass the exhausted all-rating route.
    for lang, country in LANG_COUNTRY_CONFIGS:
        for sort_mode in SORT_MODES_TO_TRY:
            for score in [1, 2, 3, 4, 5]:
                configs.append({
                    "lang": lang,
                    "country": country,
                    "sort_mode": sort_mode,
                    "filter_score_with": score,
                    "max_duplicate_batches": MAX_DUPLICATE_BATCHES_SCORE_FILTER,
                })

    # Second: all-rating routes, mostly for completeness.
    for lang, country in LANG_COUNTRY_CONFIGS:
        for sort_mode in SORT_MODES_TO_TRY:
            configs.append({
                "lang": lang,
                "country": country,
                "sort_mode": sort_mode,
                "filter_score_with": None,
                "max_duplicate_batches": MAX_DUPLICATE_BATCHES_ALL_RATINGS,
            })

    return configs


CRAWL_CONFIGS = build_crawl_configs()
print(f"Prepared {len(CRAWL_CONFIGS)} crawl route(s).")
for i, cfg in enumerate(CRAWL_CONFIGS[:10], start=1):
    print(i, cfg["lang"], cfg["country"], sort_name(cfg["sort_mode"]), cfg["filter_score_with"])
if len(CRAWL_CONFIGS) > 10:
    print("...")


Prepared 36 crawl route(s).
1 id id newest 1
2 id id newest 2
3 id id newest 3
4 id id newest 4
5 id id newest 5
6 id id most_relevant 1
7 id id most_relevant 2
8 id id most_relevant 3
9 id id most_relevant 4
10 id id most_relevant 5
...


## 5. Crawl more reviews with multi-route checkpointing

If one route starts returning only duplicates, the notebook automatically moves to the next route instead of looping forever.

If VS Code stops or your laptop restarts, run the notebook again with `RESUME = True`. It will keep the raw CSV and resume each route using its own token file when available.


In [5]:

def crawl_google_play_reviews_multi_route(
    app_id=APP_ID,
    target_additional_reviews=TARGET_ADDITIONAL_REVIEWS,
    min_total_target_reviews=MIN_TOTAL_TARGET_REVIEWS,
    batch_size=BATCH_SIZE,
):
    existing_df = load_existing_dataset(RAW_CSV_PATH)
    all_rows = existing_df.to_dict("records") if not existing_df.empty else []
    seen_hashes = set(existing_df["review_id_hash"].astype(str)) if "review_id_hash" in existing_df.columns else set()

    starting_unique = len(seen_hashes)
    target_total = max(starting_unique + int(target_additional_reviews), int(min_total_target_reviews))

    print("\n==============================")
    print("Multi-route crawling started")
    print("==============================")
    print("Starting unique reviews:", f"{starting_unique:,}")
    print("Target additional      :", f"{target_additional_reviews:,}")
    print("Target total           :", f"{target_total:,}")
    print("Routes to try          :", len(CRAWL_CONFIGS))
    print("==============================\n")

    config_stats = []
    pbar = tqdm(total=target_total, initial=min(starting_unique, target_total), desc="Collected unique reviews")

    for cfg_idx, cfg in enumerate(CRAWL_CONFIGS, start=1):
        if len(seen_hashes) >= target_total:
            break

        lang = cfg["lang"]
        country = cfg["country"]
        sort_mode = cfg["sort_mode"]
        filter_score_with = cfg["filter_score_with"]
        max_duplicate_batches = cfg["max_duplicate_batches"]

        route_name = config_name(lang, country, sort_mode, filter_score_with)
        token_path = token_path_for_config(lang, country, sort_mode, filter_score_with)
        continuation_token = load_continuation_token(token_path)

        print("\n--------------------------------------------------")
        print(f"Route {cfg_idx}/{len(CRAWL_CONFIGS)}: {route_name}")
        print(f"Sort={sort_name(sort_mode)} | Score filter={filter_score_with} | lang={lang} | country={country}")
        print(f"Duplicate batch limit for this route: {max_duplicate_batches}")
        print("Current unique:", f"{len(seen_hashes):,}", "/", f"{target_total:,}")
        print("--------------------------------------------------")

        retry_count = 0
        duplicate_batches = 0
        empty_batches = 0
        total_batches = 0
        new_from_route = 0

        while len(seen_hashes) < target_total:
            requested_count = min(batch_size, target_total - len(seen_hashes))

            try:
                batch, continuation_token = reviews(
                    app_id,
                    lang=lang,
                    country=country,
                    sort=sort_mode,
                    count=requested_count,
                    filter_score_with=filter_score_with,
                    continuation_token=continuation_token,
                )
                retry_count = 0
                total_batches += 1

            except Exception as e:
                retry_count += 1
                if retry_count > MAX_RETRIES_PER_BATCH:
                    print("Stopped this route because maximum retries were reached.")
                    print("Last error:", repr(e))
                    break

                sleep_seconds = min(MAX_BACKOFF_SECONDS, BASE_BACKOFF_SECONDS * (2 ** (retry_count - 1)))
                sleep_seconds += random.uniform(0, 3)
                print(f"Request failed. Retry {retry_count}/{MAX_RETRIES_PER_BATCH} after {sleep_seconds:.1f}s. Error: {repr(e)}")
                time.sleep(sleep_seconds)
                continue

            if not batch:
                empty_batches += 1
                print(f"No reviews returned. Empty batches: {empty_batches}/{MAX_EMPTY_BATCHES_PER_CONFIG}")
                save_checkpoint(pd.DataFrame(all_rows).drop_duplicates(subset=["review_id_hash"]), continuation_token, token_path=token_path)
                if empty_batches >= MAX_EMPTY_BATCHES_PER_CONFIG:
                    print("Moving to next route because multiple empty batches occurred.")
                    break
                time.sleep(random.uniform(MIN_DELAY_SECONDS, MAX_DELAY_SECONDS))
                continue

            empty_batches = 0
            new_rows = []

            for record in batch:
                row = normalize_review(
                    record,
                    app_id=app_id,
                    lang=lang,
                    country=country,
                    sort_mode=sort_mode,
                    filter_score_with=filter_score_with,
                )
                content = row.get("content")
                review_hash = str(row.get("review_id_hash"))

                if not content or not str(content).strip():
                    continue

                if review_hash not in seen_hashes:
                    seen_hashes.add(review_hash)
                    new_rows.append(row)

            if new_rows:
                duplicate_batches = 0
                all_rows.extend(new_rows)
                new_from_route += len(new_rows)

                current_df = pd.DataFrame(all_rows).drop_duplicates(subset=["review_id_hash"]).reset_index(drop=True)
                if "score" in current_df.columns:
                    current_df["weak_sentiment_from_star"] = current_df["score"].apply(weak_sentiment_from_star)

                save_checkpoint(current_df, continuation_token, token_path=token_path)
                pbar.update(min(len(new_rows), target_total - pbar.n))

                print(
                    f"Saved checkpoint: {len(current_df):,} unique reviews "
                    f"(+{len(new_rows):,} new from this batch, +{new_from_route:,} from route)"
                )

            else:
                duplicate_batches += 1
                if duplicate_batches == 1 or duplicate_batches % 10 == 0:
                    print(
                        f"All duplicates/empty for this batch. "
                        f"Duplicate batches: {duplicate_batches}/{max_duplicate_batches}"
                    )

                # Save token even when duplicates happen, so if the notebook stops,
                # it can resume deeper into this route instead of repeating the same duplicate pages.
                current_df = pd.DataFrame(all_rows).drop_duplicates(subset=["review_id_hash"]).reset_index(drop=True)
                save_checkpoint(current_df, continuation_token, token_path=token_path)

                if duplicate_batches >= max_duplicate_batches:
                    print("Moving to next route because duplicate batch limit was reached.")
                    break

            if continuation_token is None:
                print("No continuation token returned. Reached available end for this route.")
                break

            delay = random.uniform(MIN_DELAY_SECONDS, MAX_DELAY_SECONDS)
            time.sleep(delay)

        config_stats.append({
            "route_index": cfg_idx,
            "route_name": route_name,
            "lang": lang,
            "country": country,
            "sort_mode": sort_name(sort_mode),
            "filter_score_with": filter_score_with,
            "total_batches": total_batches,
            "duplicate_batches_at_end": duplicate_batches,
            "empty_batches_at_end": empty_batches,
            "new_reviews_from_route": new_from_route,
            "unique_reviews_after_route": len(seen_hashes),
            "finished_at_utc": datetime.now(timezone.utc).isoformat(),
        })

        # Save route summary after every route.
        with open(METADATA_PATH, "w", encoding="utf-8") as f:
            json.dump({
                "app_id": app_id,
                "starting_unique_reviews": starting_unique,
                "target_additional_reviews": int(target_additional_reviews),
                "target_total_reviews": int(target_total),
                "current_unique_reviews": int(len(seen_hashes)),
                "routes_finished": len(config_stats),
                "route_stats": config_stats,
                "raw_csv_path": str(RAW_CSV_PATH),
                "output_dir": str(OUTPUT_DIR),
                "last_updated_utc": datetime.now(timezone.utc).isoformat(),
            }, f, indent=2, ensure_ascii=False)

    pbar.close()

    final_df = pd.DataFrame(all_rows).drop_duplicates(subset=["review_id_hash"]).reset_index(drop=True)
    if "score" in final_df.columns:
        final_df["weak_sentiment_from_star"] = final_df["score"].apply(weak_sentiment_from_star)
    final_df.to_csv(RAW_CSV_PATH, index=False, encoding="utf-8-sig")

    final_metadata = {
        "app_id": app_id,
        "starting_unique_reviews": starting_unique,
        "target_additional_reviews": int(target_additional_reviews),
        "target_total_reviews": int(target_total),
        "collected_unique_reviews": int(len(final_df)),
        "added_unique_reviews_this_run": int(len(final_df) - starting_unique),
        "target_reached": bool(len(final_df) >= target_total),
        "routes_tried": len(config_stats),
        "route_stats": config_stats,
        "finished_at_utc": datetime.now(timezone.utc).isoformat(),
        "raw_csv_path": str(RAW_CSV_PATH),
        "output_dir": str(OUTPUT_DIR),
    }

    with open(METADATA_PATH, "w", encoding="utf-8") as f:
        json.dump(final_metadata, f, indent=2, ensure_ascii=False)

    print("\nFinished multi-route crawling.")
    print(json.dumps({
        "starting_unique_reviews": starting_unique,
        "collected_unique_reviews": int(len(final_df)),
        "added_unique_reviews_this_run": int(len(final_df) - starting_unique),
        "target_total_reviews": int(target_total),
        "target_reached": bool(len(final_df) >= target_total),
        "raw_csv_path": str(RAW_CSV_PATH),
        "metadata_path": str(METADATA_PATH),
    }, indent=2, ensure_ascii=False))

    if len(final_df) < target_total:
        print("\nNote: The target was not reached. That means the available Google Play routes tried here")
        print("did not expose enough new unique reviews. You can still keep the data, or add more")
        print("routes in LANG_COUNTRY_CONFIGS / SORT_MODES_TO_TRY / FILTER_SCORES_TO_TRY.")

    return final_df


if RUN_CRAWLING:
    raw_df = crawl_google_play_reviews_multi_route()
else:
    if not RAW_CSV_PATH.exists():
        raise FileNotFoundError(f"RUN_CRAWLING is False, but raw file does not exist: {RAW_CSV_PATH}")
    raw_df = pd.read_csv(RAW_CSV_PATH)
    print(f"Loaded raw file without crawling: {len(raw_df):,} rows")

display(raw_df.head())
print("Raw shape:", raw_df.shape)
if "score" in raw_df.columns:
    display(raw_df["score"].value_counts(dropna=False).sort_index())
if "source_sort_mode" in raw_df.columns:
    display(raw_df[["source_lang", "source_country", "source_sort_mode", "source_filter_score_with"]].value_counts(dropna=False).head(20))


Loaded existing checkpoint: 90,727 unique rows from C:\Users\Lenovo\Documents\Skripsii\Skripsi\dataset\shopee_google_play_reviews_raw.csv

Multi-route crawling started
Starting unique reviews: 90,727
Target additional      : 90,000
Target total           : 180,727
Routes to try          : 36



Collected unique reviews:  50%|#####     | 90727/180727 [00:00<?, ?it/s]


--------------------------------------------------
Route 1/36: com_shopee_id_id_id_newest_score_1
Sort=newest | Score filter=1 | lang=id | country=id
Duplicate batch limit for this route: 180
Current unique: 90,727 / 180,727
--------------------------------------------------
Saved checkpoint: 90,740 unique reviews (+13 new from this batch, +13 from route)
All duplicates/empty for this batch. Duplicate batches: 1/180
All duplicates/empty for this batch. Duplicate batches: 10/180
All duplicates/empty for this batch. Duplicate batches: 20/180
All duplicates/empty for this batch. Duplicate batches: 30/180
All duplicates/empty for this batch. Duplicate batches: 40/180
All duplicates/empty for this batch. Duplicate batches: 50/180
All duplicates/empty for this batch. Duplicate batches: 60/180
All duplicates/empty for this batch. Duplicate batches: 70/180
All duplicates/empty for this batch. Duplicate batches: 80/180
All duplicates/empty for this batch. Duplicate batches: 90/180
Saved checkp

,review_id_hash,content,score,thumbs_up_count,review_created_version,review_at,app_version,reply_content,replied_at,source_app_id,source_lang,source_country,crawl_timestamp_utc,source_sort_mode,source_filter_score_with,weak_sentiment_from_star
0,7f86a085947b6ae7,cantik,5,0,NaN,2026-06-05T12:12:40,NaN,"Hai kak Kuyang dead , thanks ya bintang dan ulasannya. Aku ikut senang kalo bisa terbantu dalam berbelanja online. Moga Shopee bisa terus kasih yang terbaik...",2026-06-05T15:50:07,com.shopee.id,id,id,2026-06-06T12:12:47.740473+00:00,NaN,NaN,positive
1,c8aeb95f15d820c5,👍👍👍👍👍,5,0,NaN,2026-06-05T12:12:28,NaN,"Hi kak Akhsan Arrafff, makasih ya untuk bintang dan feedbacknya. Pastinya Shopee bagus banget ya kak. Apa aja ada dan murah banget juga hihi. Moga kamu sela...",2026-06-05T15:53:48,com.shopee.id,id,id,2026-06-06T12:12:47.740517+00:00,NaN,NaN,positive
2,c96299172942e4bc,keren,5,0,3.75.28,2026-06-05T12:11:18,3.75.28,"Hi Kak Eko Prana, makasih ya buat bintang 5 nya. Kuyy pantau terus dan ceki-ceki sosmed kita biar ga ketinggalan promo menarik lainnya. Klo kamu ada pertan...",2026-06-05T15:47:15,com.shopee.id,id,id,2026-06-06T12:12:47.740531+00:00,NaN,NaN,positive
3,69d33e489f9d76ce,karna aku kena penipu 100 kali sampai aku ping dan mamaku mati kakak ku mati,1,0,NaN,2026-06-05T12:11:15,NaN,"Hai kak Al-Ghazali hasri Algazali, maaf ya udah buat kamu ngga nyaman karena kendala yg kamu alami. Biar bisa aku bantu pengecekan, kamu bisa infoin detail ...",2026-06-05T15:58:51,com.shopee.id,id,id,2026-06-06T12:12:47.740542+00:00,NaN,NaN,negative
4,8fab2c0db375a075,"Aplikasihnya bagus, cuman tolong lah kalo semisal ada paket yang di retur itu tanya ke konsumen dulu, padahal ya kurirnya sama sekali tidak datang ke rumah ...",2,0,3.75.25,2026-06-05T12:09:31,3.75.25,"Hai Kak Zynseann, makasih ya buat feedbacknya. Kedepannya Shopee akan terus berusaha lebih baik lagi agar para pengguna lebih nyaman dalam menggunakan Shope...",2026-06-05T15:47:41,com.shopee.id,id,id,2026-06-06T12:12:47.740552+00:00,NaN,NaN,negative


Raw shape: (180907, 16)


score
1    109214
2      3302
3      3456
4      4304
5     60631
Name: count, dtype: int64

source_lang  source_country  source_sort_mode  source_filter_score_with
id           id              NaN               NaN                         90727
                             newest            1.0                         90180
Name: count, dtype: int64